# பாடம் 02 - மைக்ரோசாஃப்ட் ஏஜென்ட் கட்டமைப்பை ஆராய்தல்

**மைக்ரோசாஃப்ட் ஏஜென்ட் கட்டமைப்பு (MAF)** என்பது ஔதா ஏஜென்ட்களை உருவாக்க ஒருங்கிணைந்த கட்டமைப்பு ஆகும். இது நான்கு முக்கிய கட்டளைகளுடன் ஒரு சுத்தமான, இணைத்துக்கொள்ளக்கூடிய அமைப்பை வழங்குகிறது:

- **கிளையன்ட்** – ஏ ஐ மாதிரி முடிவெடுத்த இடத்துடன் இணைந்து தொடர்பை கையாள்கிறது
- **ஏஜென்ட்** – கிளையன்டை உத்தரவுகள் மற்றும் கருவி வரையறைகளுடன் சுற்றி எடுக்கும்
- **கருவிகள்** – மாதிரி அழைக்கக்கூடிய தனிப்பட்ட செயல்பாடுகளோடு ஏஜென்ட் திறன்களை விரிவாக்குகிறது
- **அமர்வு** – பன்முறை உரையாடல்களுக்கு உரையாடல் வரலாறை பராமரிக்கிறது

இந்த பாடத்தில், நாம் இந்த கருத்துக்களை பயன்படுத்தி **பயண முன்பதிவு ஏஜென்டை** உருவாக்க போகிறோம், இது இடைநிலையை சரிபார்க்கும்.


## அமைத்தல்


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## முகவர் கட்டமைப்பு معماری-ஐ புரிந்துகொள்வது

Microsoft முகவர் கட்டமைப்பு ஒரு படிமுறை கட்டமைப்பைப் பின்பற்றுகிறது:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **கிளையன்ட்** – ஒரு `FoundryChatClient` Azure OpenAI நியமனை இணைக்கிறது. இது அங்கீகாரம், கோரிக்கை வடிவமைப்பு மற்றும் பதில் பகுப்பாய்வை செய்கிறது.
2. **முகவர்** – `provider.create_agent()` மூலம் கிளையன்டிலிருந்து உருவாக்கப்படுகிறது, முகவர் மாதிரி அணுகலை (system prompt) மற்றும் கருவிகளுடன் இணைக்கிறது.
3. **கருவிகள்** – Python செயல்பாடுகள் `@tool` மூலம் அலங்கரிக்கப்பட்டவை, முகவர் செயல்பாடுகள் செய்ய அல்லது தரவு பெற அழைக்க முடியும்.
4. **அமர்வு** – ஒரு `AgentSession` பொருள் (`agent.create_session()` மூலம் உருவாக்கப்பட்டது) உரையாடல் வரலாறை சேமித்து, முகவர் முன் உள்ள சூழலை நினைவுகொண்டு பலமுறை உரையாடல் செய்ய உதவுகிறது.

ஒவ்வொரு படியையும் படி படியாக கட்டுவோம்.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## @tool அலங்காரியைப் பயன்படுத்தி கருவிகள் சேர்ப்பது

கருவிகள் முகவர்களுக்கு உரை உருவாக்குவதுக்கு அப்பால் செயல்களை எடுக்க அனுமதிக்கின்றன. `@tool` அலங்காரி ஒரு வழக்கமான Python செயல்பாட்டை முகவர் அழைக்கக்கூடியதாக மாற்றுகிறது.

முக்கிய அம்சங்கள்:
- மாதிரி ஒவ்வொரு அளவுருவையும் புரிந்து கொள்ள `Annotated[type, "விவரம்"]` பயன்படுத்தவும்.
- ஆவணக் குறிப்பு மாதிரிக்கு காட்டப்படும் கருவி விளக்கமாக மாறும்.
- `approval_mode="never_require"` என்றால் கருவி பயனர் ஒப்புதலை வேண்டாமை அப்போதையாக இயங்கும்.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## கருவிகளுடன் ஒரு முகவரையை உருவாக்குதல்

இப்போது நாம் கிளையண்ட், வழிமுறைகள் மற்றும் கருவிகளை ஒரு முகவரியுடன் சேர்க்கிறோம். `வழிமுறைகள்` என்பது சிஸ்டம்_PROMPT ஆக செயல்படுகிறது — அவை முகவரியின் தனித்துவம் மற்றும் நடத்தையை வரையறுக்கும்.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## அமர்வுகளுடன் பல சுற்றுச்சுற்று உரையாடல்கள்

ஒரு `AgentSession` (`agent.create_session()` மூலம் உருவாக்கப்படுகிறது) உரையாடலில் உள்ள அனைத்து செய்திகள் பற்றியும் கண்காணிக்கிறது. ஒவ்வொரு `agent.run()` அழைப்புக்கும் அதே அமர்வை கொடுக்கும் மூலம், முகவர் முழு உரையாடல் வரலாற்றையும் பார்க்கக் கூடியதாயிருக்கிறார் மற்றும் முன்னணி செய்திகள் ஒன்றைப் பார்த்து மீண்டும் குறிப்பிடக்கூடியதாக இருக்கிறார்.

நாம் `tools=[check_destination_availability]` எனக் கொடுப்பதனால், முகவர் ஒவ்வொரு சுற்றிலும் எங்கள் கிடைக்கும் தன்மை சரிபார்ப்பாளரை அழைக்க முடியும்.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## சுருக்கம்

இந்த பாடத்தில் நீங்கள் Microsoft முகவர் கட்டமைப்பின் நான்கு தூண்களை ஆராய்ந்தீர்கள்:

| கருத்து | நீங்கள் கற்றுக்கொண்டது |
|---------|------------------|
| **வாடிக்கையாளர்** | `FoundryChatClient` அங்கீகார அடிப்படையிலான அங்கீகாரம் மூலம் Azure OpenAI-இல் இணைகிறது |
| **முகவர்** | `provider.create_agent()` ஒரு மாதிரி இணைப்பை, வழிமுறைகள் மற்றும் பெயருடன் தொகுத்துக் கொள்கிறது |
| **கருவிகள்** | `@tool` அலங்காரி முகவருக்காக பைதான் செயல்பாடுகளை அழைக்க வெளிப்படுத்துகிறது |
| **அமர்வு** | `agent.create_session()` பல சுற்றங்களில் உரையாடல் வரலாறை பராமரிக்கிறது |

இந்த கட்டுமானக் கூறுகள் ஒன்றிணைந்து இயற்கை உரையாடல்களை நடத்தக்கூடிய, வெளிப்புற செயல்பாடுகளை அழைக்கக்கூடிய மற்றும் சூழலை பராமரிக்கக்கூடிய முகவர்களை உருவாக்குகின்றன — பின்னர் பாடங்களில் மேம்பட்ட முகவர் மாதிரிகளுக்கான அடித்தளம்.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
